# Step 2b: LLM Evaluation of Query-Product Pairs

This notebook evaluates query-product pairs where the query has negative intent to determine if the product violates the constraint.

**Prerequisites**: Run 02a_negative_intent_detection.ipynb first to generate the negative intent mapping.
**Output**: LLM judgments (VIOLATION/COMPLIANT/UNCLEAR) merged back into search result CSV files.

In [1]:
import pandas as pd
import json
import os
import boto3
import time
from datetime import datetime
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import hashlib
from collections import defaultdict

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [2]:
%env AWS_PROFILE = worker_discovery_dev

env: AWS_PROFILE=worker_discovery_dev


## Configuration

In [3]:
# Configuration
SEARCH_RESULTS_DIR = "./search_results/"

# Separate directories for different evaluation outputs
NEGATIVE_INTENT_DIR = "./llm_evaluations/negative_intent"
PAIR_EVALUATIONS_DIR = "./llm_evaluations/pair_judgments"
MERGED_RESULTS_DIR = "./llm_evaluations/merged_csv_results"

# Create directories
os.makedirs(NEGATIVE_INTENT_DIR, exist_ok=True)
os.makedirs(PAIR_EVALUATIONS_DIR, exist_ok=True)
os.makedirs(MERGED_RESULTS_DIR, exist_ok=True)

# AWS Bedrock configuration  
model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
MAX_WORKERS = 5  # Concurrent LLM requests
BATCH_SIZE = 100  # Save checkpoints every N evaluations

# Initialize Bedrock client
bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

print(f"Search results directory: {SEARCH_RESULTS_DIR}")
print(f"Negative intent directory: {NEGATIVE_INTENT_DIR}")
print(f"Pair evaluations directory: {PAIR_EVALUATIONS_DIR}")
print(f"Merged results directory: {MERGED_RESULTS_DIR}")
print(f"Bedrock model: {model_id}")
print(f"Max concurrent workers: {MAX_WORKERS}")

Search results directory: ./search_results/
Negative intent directory: ./llm_evaluations/negative_intent
Pair evaluations directory: ./llm_evaluations/pair_judgments
Merged results directory: ./llm_evaluations/merged_csv_results
Bedrock model: us.anthropic.claude-3-7-sonnet-20250219-v1:0
Max concurrent workers: 5


## Load Prerequisites

In [4]:
# Load the negative intent detection results (manual fix version)
negative_intent_file = os.path.join(NEGATIVE_INTENT_DIR, 'negative_intent_detection_manual_fix.json')

if not os.path.exists(negative_intent_file):
    raise FileNotFoundError(f"""
    Negative intent detection file not found: {negative_intent_file}
    
    Please run 02a_negative_intent_detection.ipynb first to generate the negative intent mapping.
    """)

with open(negative_intent_file, 'r') as f:
    negative_intent_map = json.load(f)

# Filter for only queries with negative intent
queries_with_negative_intent = {k: v for k, v in negative_intent_map.items() if v is not None}

print(f"Loaded negative intent mapping:")
print(f"  Total queries: {len(negative_intent_map)}")
print(f"  Queries with negative intent: {len(queries_with_negative_intent)}")

if len(queries_with_negative_intent) == 0:
    raise ValueError("No queries with negative intent found. Please check the negative intent detection results.")

# Show some examples
print(f"\nExample queries with negative intent:")
for i, (query, intent) in enumerate(list(queries_with_negative_intent.items())[:5]):
    print(f"  {i+1}. '{query}' → excludes '{intent}'")

Loaded negative intent mapping:
  Total queries: 1000
  Queries with negative intent: 790

Example queries with negative intent:
  1. 'chicken free dry dog food' → excludes 'chicken'
  2. 'grain free dog food' → excludes 'grain'
  3. 'grain free dog treats' → excludes 'grain'
  4. 'grain free dry cat food' → excludes 'grain'
  5. 'grain free wet cat food' → excludes 'grain'


## Load and Filter Search Results

In [5]:
def load_all_search_results():
    """Load all search result CSV files from all subdirectories and combine into structured data"""
    
    # Check if directory exists
    if not os.path.exists(SEARCH_RESULTS_DIR):
        logger.error(f"Directory not found: {SEARCH_RESULTS_DIR}")
        return pd.DataFrame()
    
    all_results = []
    csv_files = []
    
    # Walk through all subdirectories to find CSV files
    for root, dirs, files in os.walk(SEARCH_RESULTS_DIR):
        for file in files:
            if file.endswith('.csv'):
                full_path = os.path.join(root, file)
                # Get relative path from search_results directory
                rel_path = os.path.relpath(full_path, SEARCH_RESULTS_DIR)
                csv_files.append((full_path, rel_path))
    
    print(f"Search directory: {SEARCH_RESULTS_DIR}")
    print(f"Found {len(csv_files)} CSV files:")
    for full_path, rel_path in csv_files:
        print(f"  {rel_path}")
    
    # Load each CSV file
    for full_path, rel_path in tqdm(csv_files, desc="Loading CSV files"):
        try:
            df = pd.read_csv(full_path)
            df['source_file'] = rel_path  # Include subdirectory in source file name
            df['subdirectory'] = os.path.dirname(rel_path) if os.path.dirname(rel_path) else 'root'
            all_results.append(df)
                    
        except Exception as e:
            logger.error(f"Error loading {rel_path}: {e}")
            continue
    
    # Combine all CSV files into one DataFrame
    if all_results:
        combined_df = pd.concat(all_results, ignore_index=True)
        return combined_df
    else:
        return pd.DataFrame()

def create_unique_evaluation_pairs(df, negative_intent_map):
    """Create unique search_term + part_number pairs for LLM evaluation, filtered for negative intent queries"""
    
    # Filter for queries with negative intent
    df_filtered = df[df['search_term'].isin(queries_with_negative_intent.keys())].copy()
    print(f"Filtered to {len(df_filtered)} results from queries with negative intent")
    
    # Remove rows with missing part_number
    df_clean = df_filtered.dropna(subset=['part_number']).copy()
    print(f"After removing missing part_numbers: {len(df_clean)} results")
    
    # Create unique pairs - keep first occurrence of each search_term + part_number combination
    unique_pairs = df_clean.drop_duplicates(subset=['search_term', 'part_number'], keep='first').copy()
    
    # Add negative intent information
    unique_pairs['negative_intent'] = unique_pairs['search_term'].map(negative_intent_map)
    
    # Create unique identifier for each pair
    unique_pairs['pair_id'] = unique_pairs.apply(
        lambda row: hashlib.md5(f"{row['search_term']}_{row['part_number']}".encode()).hexdigest()[:12], 
        axis=1
    )
    
    return unique_pairs[['search_term', 'part_number', 'sku_name', 'negative_intent', 'pair_id']]

# Load search results
df_all_results = load_all_search_results()
print(f"\nLoaded {len(df_all_results)} total search result entries")

# Create evaluation pairs
df_unique_pairs = create_unique_evaluation_pairs(df_all_results, negative_intent_map)

print(f"\n=== EVALUATION SCOPE ===")
print(f"Unique evaluation pairs: {len(df_unique_pairs)}")
print(f"Unique queries with negative intent: {df_unique_pairs['search_term'].nunique()}")
print(f"Unique products: {df_unique_pairs['part_number'].nunique()}")

# Show sample pairs
print(f"\nSample evaluation pairs:")
for i, row in df_unique_pairs.head(5).iterrows():
    print(f"  '{row['search_term']}' + {row['part_number']} (excludes '{row['negative_intent']}') → '{row['sku_name'][:50]}...'")

Search directory: ./search_results/
Found 8 CSV files:
  old_control_before_revenue_func/l1_hybrid_search_results.csv
  old_control_before_revenue_func/l2_search_results.csv
  old_control_before_revenue_func/knn_search_results.csv
  old_control_before_revenue_func/lexical_search_results.csv
  control/l1_hybrid_search_results.csv
  control/l2_search_results.csv
  control/knn_search_results.csv
  control/lexical_search_results.csv


Loading CSV files: 100%|██████████| 8/8 [00:00<00:00, 23.05it/s]



Loaded 277497 total search result entries
Filtered to 222607 results from queries with negative intent
After removing missing part_numbers: 222607 results

=== EVALUATION SCOPE ===
Unique evaluation pairs: 67302
Unique queries with negative intent: 790
Unique products: 14718

Sample evaluation pairs:
  'chicken free dry dog food' + 150190 (excludes 'chicken') → 'True Acre Foods Grain-Free Chicken & Vegetable Dry...'
  'chicken free dry dog food' + 253856 (excludes 'chicken') → 'Merrick Real Chicken + Sweet Potato Recipe Grain-F...'
  'chicken free dry dog food' + 51515 (excludes 'chicken') → 'Nature's Recipe Small Breed Grain-Free Chicken, Sw...'
  'chicken free dry dog food' + 203092 (excludes 'chicken') → 'Merrick Limited Ingredient Diet Grain-Free Chicken...'
  'chicken free dry dog food' + 51500 (excludes 'chicken') → 'Nature's Recipe Grain-Free Chicken, Sweet Potato &...'


## LLM Evaluation Functions

In [6]:
def create_negative_intent_prompt(query, sku_name, negative_intent):
    """Create prompt for negative intent violation detection"""
    
    prompt = f"""
You are evaluating pet products on Chewy.com to determine if they violate a customer's negative intent constraint.

CONTEXT: Chewy.com is a pet retailer selling food, treats, toys, supplies, and health products for dogs, cats, and other pets.

SEARCH QUERY: "{query}"
NEGATIVE INTENT: The customer wants to EXCLUDE products with "{negative_intent}"

PRODUCT INFORMATION:
Product Name: {sku_name}

TASK: Determine if this pet product violates the negative intent constraint.

EVALUATION CRITERIA:
- VIOLATION: Product contains or has the attribute that the customer wants to exclude
- COMPLIANT: Product does NOT contain the excluded attribute and meets the customer's intent  
- UNCLEAR: Insufficient product information to determine compliance

Consider common pet product attributes like:
- Ingredients (grain, chicken, beef, corn, soy, etc.)
- Features (pull, slip, noise, mess, etc.)
- Materials (rawhide, plastic, fabric, etc.)
- Allergens and sensitivities

RESPONSE FORMAT:
Provide only one word: VIOLATION, COMPLIANT, or UNCLEAR

RESPONSE:""".strip()
    
    return prompt

def call_bedrock_llm(prompt, max_tokens=10):
    """Call Bedrock LLM with error handling"""
    try:
        # Check if bedrock client exists
        if 'bedrock' not in globals() or bedrock is None:
            raise Exception("Bedrock client not initialized. Run the configuration cells first.")
            
        body = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": max_tokens,
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }
        
        response = bedrock.invoke_model(
            modelId=model_id,
            body=json.dumps(body)
        )
        
        response_data = json.loads(response['body'].read())
        return response_data['content'][0]['text'].strip().upper()
        
    except Exception as e:
        logger.error(f"Bedrock call failed: {e}")
        return "ERROR"

def evaluate_single_pair(row):
    """Evaluate a single query-product pair"""
    try:
        prompt = create_negative_intent_prompt(
            row['search_term'], 
            row['sku_name'], 
            row['negative_intent']
        )
        
        judgment = call_bedrock_llm(prompt)
        
        # Validate judgment
        if judgment not in ['VIOLATION', 'COMPLIANT', 'UNCLEAR']:
            logger.warning(f"Unexpected judgment: {judgment} for pair {row['pair_id']}")
            judgment = 'UNCLEAR'
        
        return {
            'pair_id': row['pair_id'],
            'search_term': row['search_term'],
            'part_number': row['part_number'],
            'sku_name': row['sku_name'],
            'negative_intent': row['negative_intent'],
            'judgment': judgment,
            'timestamp': datetime.now().isoformat()
        }
        
    except Exception as e:
        logger.error(f"Error evaluating pair {row.get('pair_id', 'unknown')}: {e}")
        return {
            'pair_id': row.get('pair_id', 'unknown'),
            'search_term': row.get('search_term', ''),
            'part_number': row.get('part_number', ''),
            'judgment': 'ERROR',
            'error': str(e),
            'timestamp': datetime.now().isoformat()
        }

print("LLM evaluation functions ready")

LLM evaluation functions ready


## Test Single Evaluation

Let's test the evaluation on one pair to verify everything works correctly before running the full batch.

In [7]:
# Test with a single evaluation pair
if len(df_unique_pairs) > 0:
    # Select a sample pair for testing
    test_row = df_unique_pairs.iloc[0]
    
    print("=== TESTING SINGLE EVALUATION ===")
    print(f"Query: '{test_row['search_term']}'")
    print(f"Product: '{test_row['sku_name']}'")
    print(f"Negative Intent: '{test_row['negative_intent']}'")
    print(f"Pair ID: {test_row['pair_id']}")
    
    # Show the prompt that will be sent
    test_prompt = create_negative_intent_prompt(
        test_row['search_term'], 
        test_row['sku_name'], 
        test_row['negative_intent']
    )
    
    print(f"\n=== PROMPT TO BE SENT ===")
    print(test_prompt)
    
    # Run the evaluation
    print(f"\n=== RUNNING TEST EVALUATION ===")
    test_result = evaluate_single_pair(test_row)
    
    print(f"\n=== EVALUATION RESULT ===")
    for key, value in test_result.items():
        if key == 'error':
            print(f"{key}: {value}")
        elif key == 'timestamp':
            print(f"{key}: {value}")
        else:
            print(f"{key}: {value}")
    
    # Interpret the result
    print(f"\n=== INTERPRETATION ===")
    if test_result['judgment'] == 'VIOLATION':
        print("✅ The product VIOLATES the negative intent constraint")
        print("   → The product contains the attribute the customer wants to exclude")
    elif test_result['judgment'] == 'COMPLIANT':
        print("✅ The product is COMPLIANT with the negative intent constraint") 
        print("   → The product does NOT contain the excluded attribute")
    elif test_result['judgment'] == 'UNCLEAR':
        print("⚠️ The evaluation result is UNCLEAR")
        print("   → Insufficient product information to determine compliance")
    elif test_result['judgment'] == 'ERROR':
        print("❌ ERROR occurred during evaluation")
        print("   → Check the error details above")
    else:
        print(f"⚠️ Unexpected judgment: {test_result['judgment']}")
    
    print(f"\n=== READY TO PROCEED ===")
    print("If the test looks good, proceed to run the full evaluation batch.")
    print("If not, check the prompt, model configuration, or debug the issue.")
    
else:
    print("❌ No evaluation pairs available for testing")

INFO:botocore.tokens:Loading cached SSO token for worker_discovery_dev


=== TESTING SINGLE EVALUATION ===
Query: 'chicken free dry dog food'
Product: 'True Acre Foods Grain-Free Chicken & Vegetable Dry Dog Food, 40-lb bag'
Negative Intent: 'chicken'
Pair ID: 5991b54748d2

=== PROMPT TO BE SENT ===
You are evaluating pet products on Chewy.com to determine if they violate a customer's negative intent constraint.

CONTEXT: Chewy.com is a pet retailer selling food, treats, toys, supplies, and health products for dogs, cats, and other pets.

SEARCH QUERY: "chicken free dry dog food"
NEGATIVE INTENT: The customer wants to EXCLUDE products with "chicken"

PRODUCT INFORMATION:
Product Name: True Acre Foods Grain-Free Chicken & Vegetable Dry Dog Food, 40-lb bag

TASK: Determine if this pet product violates the negative intent constraint.

EVALUATION CRITERIA:
- VIOLATION: Product contains or has the attribute that the customer wants to exclude
- COMPLIANT: Product does NOT contain the excluded attribute and meets the customer's intent  
- UNCLEAR: Insufficient prod

## Run LLM Evaluations

In [8]:
def run_llm_evaluations(df_eval, max_workers=MAX_WORKERS, batch_size=BATCH_SIZE):
    """Run LLM evaluations with checkpointing"""
    
    # Check for existing progress
    checkpoint_file = os.path.join(PAIR_EVALUATIONS_DIR, 'evaluations_checkpoint.json')
    completed_evaluations = []
    completed_pairs = set()
    
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'r') as f:
            completed_evaluations = json.load(f)
        completed_pairs = {eval_data['pair_id'] for eval_data in completed_evaluations}
        print(f"Resuming from checkpoint: {len(completed_evaluations)} evaluations already completed")
    
    # Filter remaining pairs
    remaining_pairs = df_eval[~df_eval['pair_id'].isin(completed_pairs)]
    print(f"Remaining pairs to evaluate: {len(remaining_pairs)}")
    
    if len(remaining_pairs) == 0:
        print("✅ All evaluations already completed!")
        return completed_evaluations
    
    # Run evaluations
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        
        for _, row in remaining_pairs.iterrows():
            future = executor.submit(evaluate_single_pair, row)
            futures.append(future)
        
        # Process results with progress bar
        with tqdm(total=len(futures), desc="Running evaluations") as pbar:
            for i, future in enumerate(as_completed(futures)):
                try:
                    result = future.result()
                    completed_evaluations.append(result)
                    
                    # Save checkpoint every batch_size evaluations
                    if (len(completed_evaluations) % batch_size) == 0:
                        with open(checkpoint_file, 'w') as f:
                            json.dump(completed_evaluations, f, indent=2)
                        logger.info(f"Checkpoint saved: {len(completed_evaluations)} evaluations")
                    
                except Exception as e:
                    logger.error(f"Future failed: {e}")
                
                pbar.update(1)
                
                # Small delay to avoid rate limiting
                time.sleep(0.1)
    
    # Final save
    with open(checkpoint_file, 'w') as f:
        json.dump(completed_evaluations, f, indent=2)
    
    return completed_evaluations

# Run evaluations
start_time = time.time()
print("=== STARTING LLM EVALUATIONS ===")
all_evaluations = run_llm_evaluations(df_unique_pairs)
end_time = time.time()

print(f"\n=== EVALUATION COMPLETE ===")
print(f"Total evaluations: {len(all_evaluations)}")
print(f"Total time: {end_time - start_time:.2f} seconds")
if len(all_evaluations) > 0:
    print(f"Average time per evaluation: {(end_time - start_time) / len(all_evaluations):.2f} seconds")

=== STARTING LLM EVALUATIONS ===
Resuming from checkpoint: 62900 evaluations already completed
Remaining pairs to evaluate: 4802


Running evaluations: 100%|██████████| 4802/4802 [18:13<00:00,  4.39it/s]



=== EVALUATION COMPLETE ===
Total evaluations: 67702
Total time: 1094.23 seconds
Average time per evaluation: 0.02 seconds


## Process and Save Results

In [9]:
# Convert to DataFrame for analysis
df_evaluations = pd.DataFrame(all_evaluations)

if len(df_evaluations) > 0:
    # Judgment distribution
    print("=== JUDGMENT DISTRIBUTION ===")
    judgment_counts = df_evaluations['judgment'].value_counts()
    print(judgment_counts)
    print(f"\nPercentages:")
    for judgment, count in judgment_counts.items():
        print(f"  {judgment}: {count/len(df_evaluations)*100:.1f}%")

    # Breakdown by negative intent type
    print(f"\n=== VIOLATION RATE BY NEGATIVE INTENT ===")
    violation_by_intent = df_evaluations[df_evaluations['judgment'].isin(['VIOLATION', 'COMPLIANT'])].groupby('negative_intent')['judgment'].apply(lambda x: (x == 'VIOLATION').mean() * 100)
    print(violation_by_intent.sort_values(ascending=False))

    # Save final results
    results_file = os.path.join(PAIR_EVALUATIONS_DIR, 'final_evaluations.json')
    with open(results_file, 'w') as f:
        json.dump(all_evaluations, f, indent=2)

    results_csv = os.path.join(PAIR_EVALUATIONS_DIR, 'final_evaluations.csv')
    df_evaluations.to_csv(results_csv, index=False)

    print(f"\n=== RESULTS SAVED ===")
    print(f"JSON: {results_file}")
    print(f"CSV: {results_csv}")
else:
    print("⚠️ No evaluations to process")

=== JUDGMENT DISTRIBUTION ===
judgment
COMPLIANT    40614
VIOLATION    19398
UNCLEAR       7690
Name: count, dtype: int64

Percentages:
  COMPLIANT: 60.0%
  VIOLATION: 28.7%
  UNCLEAR: 11.4%

=== VIOLATION RATE BY NEGATIVE INTENT ===
negative_intent
kibble        100.000000
catnip        100.000000
yeast         100.000000
grain, fat    100.000000
additives     100.000000
                 ...    
tip             3.401361
gmo             2.262443
toxic           0.000000
corn, soy       0.000000
iodine          0.000000
Name: judgment, Length: 102, dtype: float64

=== RESULTS SAVED ===
JSON: ./llm_evaluations/pair_judgments/final_evaluations.json
CSV: ./llm_evaluations/pair_judgments/final_evaluations.csv


## Merge Results Back to CSV Files

In [10]:
df_evaluations.head()

,pair_id,search_term,part_number,sku_name,negative_intent,judgment,timestamp
0,5991b54748d2,chicken free dry dog food,150190,True Acre Foods Grain-Free Chicken & Vegetable...,chicken,VIOLATION,2026-02-02T11:32:26.205295
1,d624f9ada948,chicken free dry dog food,51515,Nature's Recipe Small Breed Grain-Free Chicken...,chicken,VIOLATION,2026-02-02T11:32:26.208195
2,b559e5c0eae2,chicken free dry dog food,51500,"Nature's Recipe Grain-Free Chicken, Sweet Pota...",chicken,VIOLATION,2026-02-02T11:32:27.130741
3,4802446f1549,chicken free dry dog food,203092,Merrick Limited Ingredient Diet Grain-Free Chi...,chicken,COMPLIANT,2026-02-02T11:32:27.173635
4,919dd55c4c78,chicken free dry dog food,253856,Merrick Real Chicken + Sweet Potato Recipe Gra...,chicken,VIOLATION,2026-02-02T11:32:27.601838


In [11]:
def merge_llm_results_to_csv_files():
    """Merge LLM evaluation results to new CSV files in merged_results directory"""
    
    if len(df_evaluations) == 0:
        print("No evaluations to merge back")
        return
    
    # Create lookup dictionary for LLM results
    llm_lookup = df_evaluations.set_index(['search_term', 'part_number'])[['negative_intent', 'judgment']].to_dict('index')
    
    # Find all CSV files in the search results directory
    csv_files = []
    for root, dirs, files in os.walk(SEARCH_RESULTS_DIR):
        for file in files:
            if file.endswith('.csv'):
                csv_files.append(os.path.join(root, file))
    
    print(f"Merging results to {len(csv_files)} CSV files in {MERGED_RESULTS_DIR}...")
    
    # Clear and recreate the merged results directory to ensure clean state
    import shutil
    if os.path.exists(MERGED_RESULTS_DIR):
        shutil.rmtree(MERGED_RESULTS_DIR)
    os.makedirs(MERGED_RESULTS_DIR, exist_ok=True)
    
    for source_filepath in tqdm(csv_files, desc="Creating merged CSV files"):
        try:
            df = pd.read_csv(source_filepath)
            
            # Add LLM evaluation columns if they don't exist
            if 'negative_intent' not in df.columns:
                df['negative_intent'] = None
            if 'llm_judgment' not in df.columns:
                df['llm_judgment'] = None
            
            # Fill in LLM results where available
            for idx, row in df.iterrows():
                key = (row['search_term'], row['part_number'])
                if key in llm_lookup:
                    df.loc[idx, 'negative_intent'] = llm_lookup[key]['negative_intent']
                    df.loc[idx, 'llm_judgment'] = llm_lookup[key]['judgment']
            
            # Create destination path preserving directory structure
            rel_path = os.path.relpath(source_filepath, SEARCH_RESULTS_DIR)
            dest_filepath = os.path.join(MERGED_RESULTS_DIR, rel_path)
            
            # Create subdirectories if needed
            dest_dir = os.path.dirname(dest_filepath)
            if dest_dir:
                os.makedirs(dest_dir, exist_ok=True)
            
            # Save merged CSV to new location
            df.to_csv(dest_filepath, index=False)
            logger.info(f"Created merged file: {dest_filepath}")
            
        except Exception as e:
            logger.error(f"Error processing {source_filepath}: {e}")
    
    print(f"✅ Created merged CSV files in {MERGED_RESULTS_DIR}")
    print(f"   Original files in {SEARCH_RESULTS_DIR} remain unchanged")

# Perform the merge
merge_llm_results_to_csv_files()

print(f"\n=== PROCESS COMPLETE ===")
print(f"LLM evaluations completed and merged back to search result CSV files")
print(f"Ready for Step 3: Metrics Calculation")

Merging results to 8 CSV files in ./llm_evaluations/merged_csv_results...


Creating merged CSV files:   0%|          | 0/8 [00:00<?, ?it/s]

INFO:__main__:Created merged file: ./llm_evaluations/merged_csv_results/old_control_before_revenue_func/l1_hybrid_search_results.csv
Creating merged CSV files: 100%|██████████| 8/8 [00:29<00:00,  3.67s/it]

✅ Created merged CSV files in ./llm_evaluations/merged_csv_results
   Original files in ./search_results/ remain unchanged

=== PROCESS COMPLETE ===
LLM evaluations completed and merged back to search result CSV files
Ready for Step 3: Metrics Calculation
